In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import os
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score
from sklearn.model_selection import KFold
import pandas as pd

from Dataloader_Excel import read_excel_eeg, read_excel_nirs
from TemporalHybridTransformer import HybridTransformer as TMMF
from SpatialHybridTransformer import HybridTransformer as SMMF
from TSMMF import HybridTransformer as TSMMF

# Hyperparameters
depth = 4
depth_series = [4, 4] # dedicated for TSMMF
query_size = 64
key_size = 64
value_size = 64
emb_size = 64
num_heads = 4
expansion = 2
conv_dropout = 0.3
self_dropout = 0.3
cross_dropout = 0.3
cls_dropout = 0.5
num_classes = 2 # number of paradigm
batch_size = 16
num_epochs = 200
learning_rate = 0.001
model_flag = 0 # 0, 1, 2 = Temporal, Spatial, Both

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create directories for results
os.makedirs("Results", exist_ok=True)

# Initialize model inside the fold loop to prevent data leak
if not model_flag:
    model = TMMF(
    depth, query_size, key_size, value_size, emb_size, num_heads, expansion,
    conv_dropout, self_dropout, cross_dropout, cls_dropout, num_classes, device).to(device)
elif model_flag == 1:
    model = SMMF(
    depth, query_size, key_size, value_size, emb_size, num_heads, expansion,
    conv_dropout, self_dropout, cross_dropout, cls_dropout, num_classes, device).to(device)
else:
    model = TSMMF(
    depth_series, query_size, key_size, value_size, emb_size, num_heads, expansion,
    conv_dropout, self_dropout, cross_dropout, cls_dropout, num_classes, device).to(device)

# model was converted to torch.float64 to avoid overflow ealier
model = model.to(torch.float64)

# Loss function & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

def log_results(subject_id, acc):
    """Logs training loss and accuracy to a file."""
    log_path = f"Results/log_s{subject_id:02d}.txt"
    with open(log_path, "a") as log_file:
        log_file.write(f'{acc:4f}\n')
log_excel = 'Results/Log.xlsx'

def evaluate_model(eval_loader, model):
    model.eval()
    total_loss, total_correct = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for eval_eeg, eval_nirs, eval_labels in eval_loader:
            eval_eeg, eval_nirs, eval_labels = eval_eeg.to(device), eval_nirs.to(device), eval_labels.to(device)
            eval_eeg = eval_eeg.to(torch.float64)
            eval_nirs = eval_nirs.to(torch.float64)

            outputs = model(eval_eeg, eval_nirs)
            loss = criterion(outputs, eval_labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            total_correct += (preds == eval_labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(eval_labels.cpu().numpy())

    loss = total_loss / len(eval_loader)
    acc = total_correct / len(eval_loader.dataset)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    kappa = cohen_kappa_score(all_labels, all_preds)

    print(f"Evaluation Loss: {loss:.2f} | Evaluation Acc: {acc:.2f}")
    return acc, precision, recall, f1, kappa

for subject_id in range(29):
    subject_id += 1 # (1, 29) from (0, 28)

    # dataset: TMVED-3: sad, neutral and happy
    # eeg, nirs, labels = read_single_data(subject_id)

    # dataset: left and right hand motor imagery / mental arithmetic
    mode = 1 # 0 for MI, 1 for MA
    eeg, labels = read_excel_eeg(subject_id, mode) # (60, 30, 4000), (60,)
    nirs, _ = read_excel_nirs(subject_id, mode) # (60, 36, 200)

    eeg, nirs, labels = eeg.to(device), nirs.to(device), labels.to(device)

    train_size = int(0.6 * len(eeg)) # 60/40 for training/testing
    eval_size = len(eeg) - train_size

    # create a tensor dataset
    dataset = torch.utils.data.TensorDataset(eeg, nirs, labels)

    # randomly split the data into training and evaluation sets
    train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)

    acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        total_correct, total_loss = 0, 0

        for batch_eeg, batch_nirs, batch_labels in train_loader:
            batch_eeg = batch_eeg.to(device, dtype=torch.float64)
            batch_nirs = batch_nirs.to(device, dtype=torch.float64)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()
            outputs = model(batch_eeg, batch_nirs)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            total_correct += (preds == batch_labels).sum().item()

        # len(train_loader) is the number of batches
        # len(train_dataset) is the number of trials
        loss = total_loss / len(train_loader)
        acc = total_correct / len(train_loader.dataset)
        print(f"Epoch {epoch} Training Loss: {loss:.2f} | Training Acc: {acc:.2f}")

        acc, precision, recall, f1, kappa = evaluate_model(eval_loader, model)
        acc_list.append(acc)
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        kappa_list.append(kappa)

    mean_acc, std_acc = np.mean(acc_list[-50:]), np.std(acc_list[-50:])
    best_acc = np.max(acc_list)
    mean_precision, std_precision = np.mean(precision_list[-50:]), np.std(precision_list[-50:])
    mean_recall, std_recall = np.mean(recall_list[-50:]), np.std(recall_list[-50:])
    mean_f1, std_f1 = np.mean(f1_list[-50:]), np.std(f1_list[-50:])
    mean_kappa = np.mean(kappa_list[-50:])

    mean_acc, std_acc = mean_acc*100, std_acc*100
    mean_precision, std_precision = mean_precision*100, std_precision*100
    mean_recall, std_recall = mean_recall*100, std_recall*100
    mean_f1, std_f1 = mean_f1*100, std_f1*100

    log_path = f'Results/log_s{subject_id:2d}.txt'
    with open(log_path, 'a') as log_file:
        log_file.write(f'Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')
        log_file.write(f'Precision: {mean_precision:.2f} ± {std_precision:.2f}\n')
        log_file.write(f'Recall: {mean_recall:.2f} ± {std_recall:.2f}\n')
        log_file.write(f'F1: {mean_f1:.2f} ± {std_f1:2f}\n')
        log_file.write(f'Kappa: {mean_kappa:.2f}\n')
        log_file.write('\n')

    new_row = pd.DataFrame([[mean_acc, std_acc, best_acc, mean_precision, std_precision, mean_recall, std_recall, mean_f1, std_f1, mean_kappa]], 
                           columns=['Accuracy', 'Std_Accuracy', 'Best_Accurarcy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
    new_row = new_row.round(2)
    if not os.path.exists(log_excel):
        new_row.to_excel(log_excel, index=False, engine='openpyxl')
    else:
        with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
            if 'Sheet1' in writer.sheets:
                startrow = writer.sheets['Sheet1'].max_row
            else:
                startrow = 0
            new_row.to_excel(writer, index=False, header=False, startrow=startrow)

    print(f'Subject:{subject_id} Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')

print('Training and evaluation completed\n')

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Start multi-head attention
1 EEG torch.Size([60, 30, 4000]) Labels torch.Size([60])
1 fNIRS torch.Size([60, 36, 200]) Labels torch.Size([60])
Epoch 0 Training Loss: 0.74 | Training Acc: 0.56
Evaluation Loss: 0.74 | Evaluation Acc: 0.46


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1 Training Loss: 0.61 | Training Acc: 0.58
Evaluation Loss: 0.69 | Evaluation Acc: 0.54


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Epoch 17 Training Loss: 0.44 | Training Acc: 0.94
Evaluation Loss: 1.86 | Evaluation Acc: 0.46


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 18 Training Loss: 0.04 | Training Acc: 1.00
Evaluation Loss: 0.96 | Evaluation Acc: 0.46


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 19 Training Loss: 0.02 | Training Acc: 1.00
Evaluation Loss: 0.70 | Evaluation Acc: 0.54
Epoch 20 Training Loss: 0.05 | Training Acc: 0.97
Evaluation Loss: 0.38 | Evaluation Acc: 0.83
Epoch 21 Training Loss: 0.04 | Training Acc: 0.97
Evaluation Loss: 0.70 | Evaluation Acc: 0.79
Epoch 22 Training Loss: 0.20 | Training Acc: 0.97
Evaluation Loss: 0.97 | Evaluation Acc: 0.79
Epoch 23 Training Loss: 0.02 | Training Acc: 1.00
Evaluation Loss: 1.55 | Evaluation Acc: 0.62
Epoch 24 Training Loss: 0.00 | Training Acc: 1.00
Evaluation Loss: 2.37 | Evaluation Acc: 0.58
Epoch 25 Training Loss: 0.00 | Training Acc: 1.00
Evaluation Loss: 2.80 | Evaluation Acc: 0.54
Epoch 26 Training Loss: 0.47 | Training Acc: 0.89
Evaluation Loss: 2.19 | Evaluation Acc: 0.58
Epoch 27 Training Loss: 0.06 | Training Acc: 1.00
Evaluation Loss: 1.44 | Evaluation Acc: 0.62
Epoch 28 Training Loss: 0.80 | Training Acc: 0.94
Evaluation Loss: 0.79 | Evaluation Acc: 0.71
Epoch 29 Training Loss: 0.05 | Training Acc: 0.97
